# Gold Layer — Dimension: Employee
## SalesFlow Data Lakehouse | Phase 5: Analytical Layer

Reads `salesflow_dev.silver.employees` (VALID records only), builds the
employee dimension with a surrogate key and concatenated full name,
and writes to `salesflow_dev.gold.dim_employee`.

**Design:**
| Column | Type | Description |
|---|---|---|
| `employee_key` | PK | MD5 surrogate key derived from `EmployeeID` |
| `employee_id` | NK | Natural key from source system |
| `full_name` | string | Concatenation of `FirstName` + `LastName` |
| `city` | string | City |
| `province` | string | Province |
| `postal_code` | string | Postal code |
| `phone` | string | Phone number |
| `birth_date` | date | Date of birth |
| `effective_date` | date | Date this record was loaded into Gold |

In [0]:
%run ../04_Utils/common_functions

## 1. Read from Silver (VALID records only)

In [0]:
from pyspark.sql.functions import current_date, col, concat_ws, to_date, substring

# Read only VALID employees from Silver
df = spark.table("salesflow_dev.silver.employees") \
          .filter(col("data_quality_status") == "VALID")

print(f"Valid records read from Silver: {df.count()}")
display(df.limit(5))

## 2. Build `full_name`
Concatenates `FirstName` and `LastName` with a space separator.  
If either part is null, `concat_ws` gracefully skips it rather than returning null.

In [0]:
# concat_ws ignores nulls — "John" + null = "John" instead of null
df = df.withColumn(
    "full_name",
    concat_ws(" ", col("FirstName"), col("LastName"))
)

## 3. Parse `birth_date`
Cast `HireDate` to proper DateType.  
Using explicit format to handle the source date format from SQL Server exports.

In [0]:
# Parse BirthDate — applying same date parsing pattern used in silver_orders
df = df.withColumn(
    "birth_date",
    to_date(substring(col("BirthDate"), 1, 10), "yyyy/MM/dd")
)

## 4. Select and Rename Columns
Rename to snake_case convention used across the Gold layer.

In [0]:
# Select only the columns needed for the dimension and rename to snake_case
df = df.select(
    col("EmployeeID").alias("employee_id"),
    col("full_name"),
    col("City").alias("city"),
    col("Province").alias("province"),
    col("PostalCode").alias("postal_code"),
    col("Phone").cast("string").alias("phone"),  # cast Long to string
    col("birth_date")
)

## 5. Add Surrogate Key
Generates `employee_key` as an MD5 hash of `employee_id`.  
The hash is deterministic — same `employee_id` always produces the same key.

In [0]:
# Add surrogate key based on natural key employee_id
df = add_surrogate_key(df, "employee", ["employee_id"])

## 6. Add Effective Date

In [0]:
# effective_date marks when this record entered the Gold layer
df = df.withColumn("effective_date", current_date())

## 7. Final Column Order
Enforce the defined schema order: PK first, NK second, attributes, metadata last.

In [0]:
# Enforce final column order as per dimension design
df = df.select(
    "employee_key",
    "employee_id",
    "full_name",
    "city",
    "province",
    "postal_code",
    "phone",
    "birth_date",
    "effective_date"
)

print(f"Total records in dimension: {df.count()}")
display(df.limit(5))

## 8. Save as Delta Table

In [0]:
# Write to Gold layer as Delta table — overwrite for first load
df.write \
  .format("delta") \
  .mode("overwrite") \
  .saveAsTable("salesflow_dev.gold.dim_employee")

print("Table saved: salesflow_dev.gold.dim_employee")

## 9. Validation

In [0]:
dim_employee = spark.table("salesflow_dev.gold.dim_employee")

# Record count
print(f"Total records: {dim_employee.count()}")

# Surrogate key uniqueness check — must be 0 duplicates
duplicate_keys = dim_employee.groupBy("employee_key").count().filter(col("count") > 1)
print(f"\nDuplicate surrogate keys (expected 0): {duplicate_keys.count()}")


# Hire date range — sanity check
print("\nHire date range:")
display(dim_employee.select("birth_date").summary("min", "max"))

# Schema
print("\nSchema:")
dim_employee.printSchema()

# Sample
print("\nFirst 5 rows:")
display(dim_employee.limit(5))